# 05 · Magic (Dunder) Methods & Operator Overloading

**Module 06 – Object-Oriented Programming** · Estimated time: 60 min

> Merges 📗 *Special Methods* (bootcamp) with 📘 *Operator Overloading* and *Magic (Dunder) Methods* (handbook), plus containers, iteration, hashing, `__call__` and attribute hooks.

### 🎯 What you will learn
"Dunder" = **d**ouble **under**score, e.g. `__init__`. They let your objects work with Python's built-in syntax and functions:

| You write | Python calls |
|---|---|
| `MyClass(...)` | `__new__`, then `__init__` |
| `str(obj)`, `print(obj)` | `__str__` |
| `repr(obj)`, REPL display | `__repr__` |
| `len(obj)` | `__len__` |
| `a + b`, `a - b`, `a * b` | `__add__`, `__sub__`, `__mul__` |
| `a == b`, `a < b` | `__eq__`, `__lt__` |
| `obj[key]`, `obj[key] = v` | `__getitem__`, `__setitem__` |
| `x in obj` | `__contains__` |
| `for x in obj` | `__iter__` |
| `obj()` | `__call__` |
| `bool(obj)`, `if obj:` | `__bool__` (or `__len__`) |
| `hash(obj)` | `__hash__` |
| `with obj:` | `__enter__`, `__exit__` |

## 1. Object creation: `__new__` and `__init__`

- `__new__(cls, ...)` **creates** the object (rarely overridden; used for immutable types and singletons)
- `__init__(self, ...)` **initialises** it

In [1]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Alice", 30)
print(p.name, p.age)

Alice 30


## 2. String representations: `__str__` vs `__repr__`

- `__str__`: **human-friendly**, informal; used by `print()` and `str()`
- `__repr__`: **unambiguous**, for developers; ideally valid Python that recreates the object; used by `repr()`, the REPL, and inside containers
- If `__str__` is missing, Python falls back to `__repr__`. ✅ **Always define at least `__repr__`.**

In [2]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"Person({self.name}, {self.age})"              # user-friendly

    def __repr__(self):
        return f"Person(name={self.name!r}, age={self.age!r})"  # for debugging

p = Person("Alice", 30)
print(str(p))
print(repr(p))
print(p)              # print() uses __str__
print([p])            # containers use __repr__
p                     # the notebook shows __repr__

Person(Alice, 30)
Person(name='Alice', age=30)
Person(Alice, 30)
[Person(name='Alice', age=30)]


Person(name='Alice', age=30)

## 3. `__len__` and `__del__` (📗 Book example)

In [3]:
class Book:
    def __init__(self, title, author, pages):
        print("A book is created")
        self.title = title
        self.author = author
        self.pages = pages

    def __str__(self):
        return f"Title: {self.title}, author: {self.author}, pages: {self.pages}"

    def __len__(self):
        return self.pages

    def __del__(self):
        print("A book is destroyed")

book = Book("Python Rocks!", "Jose Portilla", 159)
print(book)
print(len(book))
del book          # removes the name; the object is destroyed when no references remain

A book is created
Title: Python Rocks!, author: Jose Portilla, pages: 159
159
A book is destroyed


> ⚠️ `__del__` (a finaliser) is rarely a good idea: *when* it runs isn't guaranteed. Use context managers (`with`) for cleanup instead (Module 09).

## 4. Operator overloading: arithmetic (📘)

In [4]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):                 # v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):                 # v1 - v2
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):                # v * 3
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):               # 3 * v (reflected)
        return self * scalar

    def __neg__(self):                        # -v
        return Vector(-self.x, -self.y)

    def __abs__(self):                        # abs(v) → length
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def __eq__(self, other):                  # v1 == v2
        return isinstance(other, Vector) and (self.x, self.y) == (other.x, other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1, v2 = Vector(2, 3), Vector(4, 5)
print(v1 + v2)        # calls v1.__add__(v2)
print(v2 - v1)
print(v1 * 5, 5 * v1)
print(-v1, abs(Vector(3, 4)))
print(v1 == Vector(2, 3), v1 == v2)

Vector(6, 8)
Vector(2, 2)
Vector(10, 15) Vector(10, 15)
Vector(-2, -3) 5.0
True False


### Reflected (`__r*__`) and in-place (`__i*__`) operators
- `a + b` tries `a.__add__(b)`; if that returns `NotImplemented`, Python tries `b.__radd__(a)`
- `a += b` tries `a.__iadd__(b)` first, then falls back to `__add__`
- ✅ Return `NotImplemented` (not raise) for unsupported types so Python can try the other side:

In [5]:
class Money:
    def __init__(self, amount):
        self.amount = amount
    def __add__(self, other):
        if isinstance(other, Money):
            return Money(self.amount + other.amount)
        if isinstance(other, (int, float)):
            return Money(self.amount + other)
        return NotImplemented
    __radd__ = __add__                        # so sum() works (it starts from 0 + ...)
    def __repr__(self):
        return f"Money({self.amount})"

print(Money(5) + Money(10), Money(5) + 2, 2 + Money(5))
print(sum([Money(1), Money(2), Money(3)]))

Money(15) Money(7) Money(7)
Money(6)


### Arithmetic & comparison dunder reference (📘)

| Operator | Method | Operator | Method |
|---|---|---|---|
| `+` | `__add__` | `==` | `__eq__` |
| `-` | `__sub__` | `!=` | `__ne__` (auto from `__eq__`) |
| `*` | `__mul__` | `<` | `__lt__` |
| `/` | `__truediv__` | `>` | `__gt__` |
| `//` | `__floordiv__` | `<=` | `__le__` |
| `%` | `__mod__` | `>=` | `__ge__` |
| `**` | `__pow__` | `@` | `__matmul__` |

## 5. Comparisons and sorting

Define `__eq__` and `__lt__`, and let `functools.total_ordering` fill in the rest:

In [6]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, text):
        self.parts = tuple(int(p) for p in text.split("."))
    def __eq__(self, other):
        return self.parts == other.parts
    def __lt__(self, other):
        return self.parts < other.parts
    def __repr__(self):
        return "v" + ".".join(map(str, self.parts))

versions = [Version("3.10.1"), Version("3.9"), Version("3.15.0"), Version("3.14.2")]
print(sorted(versions), max(versions))
print(Version("3.9") >= Version("3.10"))       # __ge__ generated automatically

[v3.9, v3.10.1, v3.14.2, v3.15.0] v3.15.0
False


## 6. `__hash__`: usable in sets and as dict keys

If you define `__eq__`, Python sets `__hash__ = None` (your objects become unhashable) unless you also define `__hash__`. Rule: **equal objects must have equal hashes**; only hash immutable data.

In [7]:
class Point:
    def __init__(self, x, y):
        self._x, self._y = x, y
    def __eq__(self, other):
        return isinstance(other, Point) and (self._x, self._y) == (other._x, other._y)
    def __hash__(self):
        return hash((self._x, self._y))
    def __repr__(self):
        return f"Point({self._x}, {self._y})"

print({Point(1, 2), Point(1, 2), Point(3, 4)})     # duplicates removed

{Point(1, 2), Point(3, 4)}


## 7. Container behaviour: `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__`, `__len__`

In [8]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self._songs = []

    def add(self, song):
        self._songs.append(song)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, index):          # p[0], also enables slicing & iteration
        return self._songs[index]

    def __setitem__(self, index, song):    # p[0] = "x"
        self._songs[index] = song

    def __delitem__(self, index):          # del p[0]
        del self._songs[index]

    def __contains__(self, song):          # "x" in p
        return song in self._songs

    def __iter__(self):                    # for s in p
        return iter(self._songs)

    def __bool__(self):                    # if p:
        return bool(self._songs)

p = Playlist("Road trip")
for s in ["Song A", "Song B", "Song C"]:
    p.add(s)
print(len(p), p[0], p[-1], p[0:2])
p[1] = "Song B (remix)"
del p[0]
print("Song C" in p, list(p), bool(p), bool(Playlist("empty")))

3 Song A Song C ['Song A', 'Song B']
True ['Song B (remix)', 'Song C'] True False


## 8. `__call__`: objects that behave like functions

In [9]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
    def __call__(self, value):
        return value * self.factor

triple = Multiplier(3)
print(triple(10), callable(triple))
print(list(map(triple, [1, 2, 3])))

30 True
[3, 6, 9]


## 9. Attribute hooks: `__getattr__`, `__setattr__`, `__delattr__`

- `__getattr__` is called only when normal lookup **fails**, which is good for defaults and proxies
- `__setattr__` intercepts **every** assignment (be careful to avoid infinite recursion)

In [10]:
class Config:
    def __init__(self, **settings):
        object.__setattr__(self, "_data", dict(settings))

    def __getattr__(self, name):                 # only for missing attributes
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(f"No setting {name!r}") from None

    def __setattr__(self, name, value):
        print(f"[log] setting {name} = {value!r}")
        self._data[name] = value

cfg = Config(debug=True, theme="dark")
print(cfg.debug, cfg.theme)
cfg.theme = "light"
print(cfg.theme)

True dark
[log] setting theme = 'light'
light


## 10. Context manager protocol (preview)

In [11]:
class Timer:
    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc, tb):
        import time
        print(f"Elapsed: {time.perf_counter() - self.start:.4f}s")
        return False          # don't suppress exceptions

with Timer():
    sum(range(1_000_000))

Elapsed: 0.0069s


(Full lesson in Module 09 · Context Managers.)

## 📝 Summary
- Dunder methods plug your class into Python's syntax and built-ins.
- Always define `__repr__`; add `__str__` for user-facing text.
- Overload operators with `__add__`, `__eq__`, `__lt__`, ...; return `NotImplemented` for unsupported types.
- Define `__hash__` alongside `__eq__` if objects must go into sets/dicts.
- `__len__`, `__getitem__`, `__iter__`, `__contains__` make container-like classes.
- `__call__` makes instances callable.

## 🏋️ Practice
1. Create a `Fraction(num, den)` class with `+`, `*`, `==`, and a `__repr__` that simplifies (use `math.gcd`).
2. Create a `Matrix2x2` with `@` (matrix multiplication) via `__matmul__`.
3. Create a `ShoppingCart` that supports `len(cart)`, `cart["apple"]` (quantity), `"apple" in cart`, and iteration.

<details><summary>Show solutions</summary>

```python
import math
class Fraction:
    def __init__(self, num, den):
        g = math.gcd(num, den)
        self.num, self.den = num // g, den // g
    def __add__(self, o): return Fraction(self.num * o.den + o.num * self.den, self.den * o.den)
    def __mul__(self, o): return Fraction(self.num * o.num, self.den * o.den)
    def __eq__(self, o): return (self.num, self.den) == (o.num, o.den)
    def __repr__(self): return f"{self.num}/{self.den}"

class Matrix2x2:
    def __init__(self, a, b, c, d): self.m = (a, b, c, d)
    def __matmul__(self, o):
        a, b, c, d = self.m; e, f, g, h = o.m
        return Matrix2x2(a*e + b*g, a*f + b*h, c*e + d*g, c*f + d*h)
    def __repr__(self): return f"Matrix2x2{self.m}"

class ShoppingCart:
    def __init__(self): self.items = {}
    def add(self, item, qty=1): self.items[item] = self.items.get(item, 0) + qty
    def __len__(self): return sum(self.items.values())
    def __getitem__(self, item): return self.items[item]
    def __contains__(self, item): return item in self.items
    def __iter__(self): return iter(self.items)
```
</details>